# 0 — Download data from the BioImage Archive

Fetches the CellProfiler feature tables from **S-BIAD2254** and prepares for `1_FeatureSorting.ipynb`. 

The figures run from the processed profile tables in `data/`, which `utils/download_data.py` fetches in ~550 MB. This notebook is
for re-deriving those tables from the CellProfiler output.


In [ ]:
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import DATA_ROOT, metadata, require

import pandas as pd

from utils.download_data import BIA_FILES_URL, check_parquet_intact, fetch_url

PLATES = ["PB000137", "PB000138", "PB000139", "PB000140", "PB000141", "PB000142"]
OBJECTS = ["featICF_nuclei", "featICF_cells", "featICF_cytoplasm"]

# Lands under data/, which is gitignored, and is where cellprofiler_results() looks.
sourceDir = DATA_ROOT / "cellprofiler_results" / "exp1_main"
sourceDir.mkdir(parents=True, exist_ok=True)
print("downloading into", sourceDir)


def fetch_remote(row: dict) -> None:
    if not BASE_URL:
        raise SystemExit(
            "No dataset URL configured — set COLOPAINT3D_DATA_URL to the archive, a "
            "mirror, or a staging copy."
        )
    # skip_existing=False: the caller has already decided this file needs (re)fetching,
    # having checked size and sha256 against the manifest.
    fetch_url(f"{BASE_URL}/{row['path']}", DATA_ROOT / row["path"], skip_existing=False)


def fetch_url(url: str, dest: Path, skip_existing: bool = True,
              attempts: int = 4) -> Path:
    """Stream ``url`` to ``dest``, via a .part file so a kill cannot leave a truncated
    table that looks complete.

    Retries on transient network errors: the CellProfiler tier is 16.6 GB across 18
    files of ~400 MB each, and EBI drops a connection often enough that a single-shot
    fetch will not get through the set. Retries are whole-file, not ranged — the archive
    does not reliably honour Range on these objects, and a silently-resumed-wrong file
    is worse than a slow one.

    Shared with ``analysis/0_Download``, so both fetchers behave the same way and there
    is one place where download semantics live.
    """
    dest = Path(dest)
    if skip_existing and dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    req = urllib.request.Request(url, headers={"User-Agent": "colopaint3D-paper/1.0"})

    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp, tmp.open("wb") as out:
                expected = resp.headers.get("Content-Length")
                shutil.copyfileobj(resp, out, CHUNK)
            if expected is not None and tmp.stat().st_size != int(expected):
                raise OSError(f"short read: {tmp.stat().st_size} of {expected} bytes")
            tmp.replace(dest)
            return dest
        except (urllib.error.URLError, OSError, http.client.HTTPException) as exc:
            tmp.unlink(missing_ok=True)
            if attempt == attempts:
                raise SystemExit(
                    f"download failed for {url} after {attempts} attempts: {exc}")
            wait = 2 ** attempt
            print(f"    retry {attempt}/{attempts - 1} in {wait}s ({exc})", flush=True)
            time.sleep(wait)
    return dest  # unreachable



## Where each plate goes

`1_FeatureSorting` builds its per-plate path as `{sourceDir}/{barcode}/{image_id}/{cp_id}`,
taking `image_id` and `cp_id` from the shipped metadata. Read them from there rather than
inventing them, so the download lands where the next notebook will look.


In [ ]:
meta = pd.read_csv(require(metadata("spher_colo52-metadata.csv", "exp1_main")))
ids = (meta[["barcode", "image_id", "cp_id"]]
       .drop_duplicates()
       .set_index("barcode"))

missing = [bc for bc in PLATES if bc not in ids.index]
assert not missing, f"no image_id/cp_id in the shipped metadata for {missing}"

for bc in PLATES:
    print(f"  {bc} -> {ids.loc[bc, 'image_id']}/{ids.loc[bc, 'cp_id']}")

## Feature tables

16.6 GB in 18 files. Already-downloaded files are skipped, so this is safe to re-run.


In [ ]:
total = 0
bad = []
for bc in PLATES:
    dest_dir = sourceDir / bc / str(ids.loc[bc, "image_id"]) / str(ids.loc[bc, "cp_id"])
    for obj in OBJECTS:
        out = dest_dir / f"{obj}.parquet"
        had = out.exists() and out.stat().st_size > 0
        fetch_url(f"{BIA_FILES_URL}/results/{bc}/{obj}.parquet", out)
        total += out.stat().st_size

        # Verify the file is a complete parquet, not just the right number of bytes.
        # A truncated upload still serves a matching Content-Length, so size proves
        # nothing; the closing PAR1 footer does.
        complaint = check_parquet_intact(out)
        if complaint:
            bad.append((f"{bc}/{obj}", complaint))
        print(f"  {'skipped ' if had else 'fetched '} {out.relative_to(sourceDir)}"
              f"  {out.stat().st_size / 1e6:6.0f} MB  {complaint or 'ok'}")

print(f"\n{total / 1e9:.2f} GB present")
if bad:
    print(f"\n{len(bad)} of {len(PLATES) * len(OBJECTS)} files are unusable:")
    for name, why in bad:
        print(f"    {name}: {why}")
    raise SystemExit(
        "The deposited CellProfiler tables are incomplete — this is a problem with the "
        "archived files, not with the download. Re-upload them before re-running; the "
        "figures do not need them (see utils/download_data.py)."
    )

## Check

`1_FeatureSorting` picks this up on its own — `cellprofiler_results("exp1_main")` prefers
this download over the cluster mount. Nothing to edit by hand.


In [ ]:
from utils.paths import cellprofiler_results

for bc in PLATES:
    d = sourceDir / bc / str(ids.loc[bc, "image_id"]) / str(ids.loc[bc, "cp_id"])
    for obj in OBJECTS:
        assert (d / f"{obj}.parquet").stat().st_size > 0, f"empty or missing: {d/obj}"

resolved = cellprofiler_results("exp1_main")
print("all 18 parquets present")
print("1_FeatureSorting will read from:", resolved)
assert resolved == sourceDir, (
    f"resolver points at {resolved}, not the download. Unset COLOPAINT3D_CP_RESULTS.")